# From Agents to Interactive Teams  

## Build Interactive Multimodel AI Agents with AutoGen  
- Imagine AI Assistant that don't just answer your questions but can take on roles (like a Marketing Manager) communicate with each other, and work towards a common goal.
- What if these assistants could even use different AI 'brains' (like GPT for one and Gemini for another) within the same team? This is the flexibility we'll explore

![Agent Overview](Agent_overview.png)

## Lerning Objectives  
- Understandthe basic concept of multi-model AI Agents in AutoGen.
- Configure AutoGen and load API keys for OpenAI, Google Gemini, and Ollama hosted models.
- Create a single odel AI Agents including a Marketing CMO and Brand Marketer using OpenAI's GPT first.
- Make the AI agents discuss ideas using AutoGen's chat capabilities (initiate_chat, max_turns).
- Modify agents to use different multi-model LLMs (e.g., Gemini for CMO, GPT for Marketer, ollama Hosted model for Brand Manager/Social media account manager etc.).
- Introduce Human Oversight: Add a "User Proxy" agent with a human in the loop.


## AutoGen 101  
### What is AutoGen?  
- AutoGen is an open-source framework that simplifies how we build and manage multi-agent systems using Large Language Models like GPT-4o, Claude, etc.
- It helps you orchestrate multiple AI agents that can automatically talk to each other, collaborate, and sovle complex tasks.  
### What is an Agent in AutoGen.  
- In AutoGen, an agent is an entity that can send and receive messages to and from other agents in its environment.
- An Agent can be powered by LLMs models, code executors (such as an IPython kernel), human, or a combination fo these and other pluggable and customizable components.

![Conversable Agent](ConversableAgent.png)


- **Documentation link:** https://microsoft.github.io/autogen/0.2/docs/tutorial/conversation-patterns


## Group Chat.  
- AutoGen provides a conversation pattern called "group chat", which involves more than two agents.
- The idea of group chat is that all agents contribute to a single conversation thread and share the same context. This is useful for tasks that require collaboration among multiple agetns.
- A group chat is orchestrated by a special agent type "GroupChatManager". At first, the Group Chat Manager selects an agent to speak. Then, the selected agent speaks, and the message is sent back to the Group Chat Manager, who broadcasts the message to all other agents in the group. This process repeats until the conversation stops.  


![Group Chat Management](GroupChat.png)

## Create an AI Agent with Similar LLM (OpenAI) First.  
We will need to install `pyautogen` and `openai`. make sure your `.env` file is in the same directory and contains OpenAI API key:

In [2]:
# Install necessary Libraries.
# Add google-generativeai for Gemini models

%pip install -q pyautogen openai python-dotenv gradio google-generativeai "ag2[gemini]"

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pyautogen (from versions: none)
ERROR: No matching distribution found for pyautogen

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Import necessary libraries
import os
from autogen_agentchat.agents import AssistantAgent
import gradio as gr
from openai import OpenAI  # Keep for reference if needed
from dotenv import load_dotenv
from IPython.display import display, Markdown
import random  # Used later for unique Gradio outputs
import google.generativeai as genai  # Import the Google library

# Load environment variables from the .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")  # Load the Google API key
ollama_api_key = os.getenv("OLLAMA_API_KEY")
print("Setup Complete: Libraries installed and API keys loaded.")

Setup Complete: Libraries installed and API keys loaded.


In [2]:
# Base Models to use OpenAIs client library
ollama_base_url = 'http://localhost:11434/v1'
gemini_base_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'


In [3]:
# Helper function to print markdown.
def print_markdown(text):
    display(Markdown(text))

Before covering AutoGen, Let's review the core concepts of an "AI Agent".  

**Analogy: A Specialized Assistant**:  
Imagine you hire two assistants for a project  
    - **Assistant A (The Planner):** Their job is to understand the *overall goal*, break it down into *steps*, and *delegate* tasks. They focus on the big picture.
    - **Assistant B (The Doer):** Their Job is to *execute specific tasks* given to them by the Planner. They focus on the details implementation.  
You wouldn't just give the Doer the final goal; you'd have the Planner instruct the Doer. They might talk back and forth.  
**AI Agents are similar:**  
- They are AI instances given a specific **Role** or **Personality**.
- They have a **Goal** or **Instructions** (defined via a 'system message').
- They can **Communicate** (send messages to each other or a human).
- They often work **Collaboratively**.
- Crucially, **they don't all need the same 'brain'**. One agent could use OpenAI's GPT, another Google's Gemini, etc., depending on what fits their role best.  
**Autogen** helps us create these assistants and manage their conversation, regardless of the underlying AI model powering them.

Let's start by creating ur Chief Marketing Officer (CMO) and Brand Marketer agents, initially having both use the same LLM (Ollama hosted model `Gemma4:e4b`) for simplicity. This helps understand the basic agent creation process.  
We need to define an `llm_config` specifiying the model and API key.

In [4]:
# Configuration for Ollama hosted agetn
config_list_ollama = [{
    'model' : "gemma4:e4b",
    'api_key' : ollama_api_key,
}
]
llm_config_ollama = {
    'config_list' : config_list_ollama,
    'temperature' : 0.7, 
    'timeout' : 5000,           # For slower local hosted model response time
}

# Configuration for OpenAI Agent
config_list_openai = [{
    'model' : 'gpt-4o-mini',
    'api_key' : openai_api_key
}]

llm_config_openai = {
    'config_list' : config_list_ollama,
    'temperature' : 0.7, 
    'timeout' : 5000,           # For slower local hosted model response time
}

In [5]:
from autogen_ext.models.ollama import OllamaChatCompletionClient

print("Import successful")

Import successful


In [6]:
cmo_prompt = """
You are the Chief Marketing Officer (CMO) of a new shoe brand (sustainable).
Provide high-level strategy, define target audiences, and guide the Marketer. Focus on the big picture. Be concise.
"""

In [7]:
brand_marketer_prompt = """
You are the Brand Marketer for the shoe brand. Brainstrogm creative, specific campaign ideas (digital, content, experience).
Focus on tactics and details. Suggest KPIs for your ideas.
"""

In [8]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini"  # or any model you have access to
)

In [ ]:
# Let's create Agents (Both using Ollama initially)
# Create the Chief Marketing Officer (CMO) Agent - Using OpenAI for now
from autogen_agentchat.agents import AssistantAgent

cmo_agent_openai = AssistantAgent(
    name="Chief_Marketing_Officer_OpenAI",
    model_client=model_client,
    system_message=cmo_prompt,
)

print(f"Agent '{cmo_agent_openai.name}' created (using OpenAI)")

Agent 'Chief_Marketing_Officer_OpenAI' created (using OpenAI)


In [11]:
# Create the Brand Marketer Agent - Using OpenAI for now
brand_marketer_agent_openai = AssistantAgent(
    name = "Brand_Marketer_OpenAI",
    model_client = model_client,
    system_message = brand_marketer_prompt,
)

print(f"Agent '{brand_marketer_agent_openai.name}' created (using OpenAI).")

Agent 'Brand_Marketer_OpenAI' created (using OpenAI).


## Test AI Agents Conversation With Similar LLM (Fun Alert!).  
Let's make our *OpenAI-only* agents talk using `initiate_chat()` and `max_turns`. This shows the fundamental conversation flow before we introduce multiple models.


In [13]:
initial_task_message = """
Context: We're launching a new sustainable shoe line and need campaign ideas.
Instruction: Brainstorm a campaign concept with specific elements.
Input: Our sustainable, futuristic shoe brand needs marketing direction.
Output: A concise campaign concept with the following structure:
Brand Marketer, let's brainstorm initial campaign ideas for our new sustainable shoe line.
Give me a distinct campaign concept.
Outline: core idea, target audience, primary channels, and 1-2 KPIs. Keep it concise. Try to arrive at a final
answer in 2-3 turns.

"""

print("--- Starting Agent Conversation (OpenAI Only) ---")
print("Chief Marketing Officer (OpenAI) initiating chat with Brand Marketer (OpenAI). Max Turns = 4")
print("---------------------------------------------------")

# Chief Marketing Officer (OpenAI) initiates the chat with Brand Marketer (OpenAI)
result = cmo_agent_openai.run(
    task = initial_task_message
)

print("---------------------------------------------")
print("--- Conversation Ended (OpenAI Only) ---")


--- Starting Agent Conversation (OpenAI Only) ---
Chief Marketing Officer (OpenAI) initiating chat with Brand Marketer (OpenAI). Max Turns = 4
---------------------------------------------------
---------------------------------------------
--- Conversation Ended (OpenAI Only) ---
